In [ ]:
# CELL 1 — Setup + train the two models
!pip install -q streamlit scikit-learn pandas numpy

import numpy as np
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier, IsolationForest

np.random.seed(42)
n = 1500

sleep_hrs = np.random.uniform(3, 10, n)
sleep_quality = np.random.randint(1, 6, n)
academic_hrs = np.random.uniform(1, 10, n)
caffeine_mg = np.random.uniform(0, 600, n)
screen_hrs = np.random.uniform(1, 14, n)
exercise_mins = np.random.uniform(0, 150, n)
mood_rating = np.random.randint(1, 6, n)
social_hrs = np.random.uniform(0, 25, n)
social_quality = np.random.randint(1, 6, n)
workload_pressure = np.random.randint(1, 6, n)

stress_score = (
    (10 - sleep_hrs) * 1.8 +
    (6 - sleep_quality) * 1.5 +
    academic_hrs * 1.2 +
    (caffeine_mg / 100) * 0.8 +
    (screen_hrs / 2) * 0.9 +
    workload_pressure * 2.0 -
    (exercise_mins / 30) * 1.2 -
    (mood_rating) * 1.5 -
    (social_hrs / 5) * 0.5 -
    (social_quality) * 1.0
)

labels = ['Low' if s < 8 else 'Moderate' if s < 18 else 'High' for s in stress_score]

df = pd.DataFrame({
    'sleep_hrs': sleep_hrs, 'sleep_quality': sleep_quality,
    'academic_hrs': academic_hrs, 'caffeine_mg': caffeine_mg,
    'screen_hrs': screen_hrs, 'exercise_mins': exercise_mins,
    'mood_rating': mood_rating, 'social_hrs': social_hrs,
    'social_quality': social_quality, 'workload_pressure': workload_pressure,
    'stress_level': labels
})

X = df.drop(columns=['stress_level'])
y = df['stress_level']

anomaly_model = IsolationForest(contamination=0.04, random_state=42).fit(X)
stress_model = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42).fit(X, y)

with open('anomaly_model.pkl', 'wb') as f:
    pickle.dump(anomaly_model, f)
with open('stress_model.pkl', 'wb') as f:
    pickle.dump(stress_model, f)

print("✅ Training complete. PKL files created!")


print("Setup and model training complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 22.4 MB/s eta 0:00:00
✅ Training complete. PKL files created!
Setup and model training complete.


In [ ]:
%%writefile chatbot.py
import random

# ============================================================
# SAHAARA — Student Stress Relief Chatbot
# ============================================================

DATA = {
    "exams": (
        [
            "Exam pressure can make the whole syllabus feel like an emergency, but you only need to handle one piece at a time.",
            "One exam is an event, not a definition of your ability.",
            "You can take academics seriously without treating every paper like a verdict on your future."
        ],
        [
            "Make a must-do / should-do / can-wait list.",
            "Choose one chapter or question type and work on it for 20–30 minutes.",
            "Use active recall or practice questions instead of only rereading.",
            "Do one timed practice set and review the mistakes.",
            "Protect your sleep instead of trying to learn everything at the last minute."
        ]
    ),

    "marks": (
        [
            "A mark is information about one assessment, not a measurement of your entire worth.",
            "It is okay to be disappointed without deciding that you are incapable.",
            "A disappointing result can become useful once you identify exactly where the marks went."
        ],
        [
            "Separate mistakes caused by knowledge gaps from careless errors.",
            "Make a short error list and revise those areas first.",
            "Compare your current work with your own previous work rather than someone else's score.",
            "Ask your teacher about anything you genuinely do not understand.",
            "Give yourself time to process the result before making big conclusions."
        ]
    ),

    "study": (
        [
            "You do not need to study perfectly. You need a method you can actually repeat.",
            "A huge syllabus becomes less scary when it becomes one small task.",
            "A short focused session is more useful than hours of panicked staring at notes."
        ],
        [
            "Write the exact topic you will finish in the next 20–30 minutes.",
            "Use questions, flashcards or active recall to check what you actually know.",
            "Keep your phone physically away during the study block.",
            "Take a proper short break instead of scrolling while pretending to study.",
            "Stop trying to make the timetable perfect and make it realistic."
        ]
    ),

    "deadlines": (
        [
            "Several deadlines together can make everything feel urgent at once.",
            "You do not have to mentally carry every unfinished task simultaneously.",
            "The goal is to reduce the pile one task at a time."
        ],
        [
            "Write every deadline down with its actual due date.",
            "Do the nearest important deadline first.",
            "Break a large assignment into the smallest possible first step.",
            "If a deadline is genuinely impossible, communicate early instead of disappearing.",
            "Submit solid work rather than exhausting yourself chasing perfection."
        ]
    ),

    "procrastination": (
        [
            "Procrastination often means the task feels too large, unclear or uncomfortable, not that you are simply lazy.",
            "You do not need motivation before starting.",
            "Make the first step so small that your brain has very little to argue with."
        ],
        [
            "Use a five-minute start.",
            "Open the exact book, document or question before starting the timer.",
            "Remove one distraction instead of trying to become perfectly disciplined.",
            "Choose one tiny completed task before planning the rest of the day.",
            "Once you start, continue only if it feels manageable."
        ]
    ),

    "focus": (
        [
            "Stress and mental overload can make concentration much harder.",
            "You do not need perfect focus; you need a short period of useful focus.",
            "If your brain is scattered, make the task smaller."
        ],
        [
            "Put the phone in another room or out of reach.",
            "Work in a short focused block followed by a planned break.",
            "Keep only the material needed for the current task visible.",
            "Write distracting thoughts on paper so you do not have to hold them in your head.",
            "Check whether poor sleep or exhaustion is making concentration worse."
        ]
    ),

    "memory": (
        [
            "Forgetting after studying does not mean you are bad at learning.",
            "Memory improves when you retrieve information instead of only rereading it.",
            "Stress and lack of sleep can make recall feel much worse."
        ],
        [
            "Close the book and explain the concept from memory.",
            "Use practice questions to find the exact gaps.",
            "Review difficult information again after a gap rather than cramming it repeatedly.",
            "Make short links between concepts instead of memorising isolated sentences.",
            "Protect sleep before an important exam."
        ]
    ),

    "parents": (
        [
            "Parent pressure can feel especially intense because their opinion matters to you.",
            "You can understand your parents' hopes without making their expectations your entire identity.",
            "You deserve support while you are figuring things out."
        ],
        [
            "Talk when everyone is calmer, not during an argument.",
            "Explain one specific effect of the pressure rather than listing everything.",
            "Show a realistic plan if your parents are worried about your work.",
            "Ask for one concrete change, such as uninterrupted study time.",
            "If conversations repeatedly become overwhelming, involve a trusted teacher or counsellor."
        ]
    ),

    "friends": (
        [
            "That sounds hurtful. Someone being mean does not automatically mean there is something wrong with you.",
            "You deserve friendships where respect goes both ways.",
            "You can care about someone and still decide that their behaviour is not okay."
        ],
        [
            "Look at the pattern rather than one isolated moment.",
            "If it feels safe, explain the specific behaviour that hurt you.",
            "Do not send a long angry message immediately; give yourself time first.",
            "Spend more time around people who treat you well.",
            "If bullying or harassment is involved, tell a trusted adult rather than handling it alone."
        ]
    ),

    "bullying": (
        [
            "Being bullied is not something you have to solve by yourself.",
            "Someone else's behaviour does not define your worth.",
            "You deserve to feel safe at school and online."
        ],
        [
            "Tell a parent, teacher, counsellor or another trusted adult.",
            "Save evidence of serious online harassment.",
            "Block, mute or report abusive accounts when appropriate.",
            "Stay around supportive people instead of isolating yourself.",
            "If you feel unsafe, prioritise getting to a safe adult or place."
        ]
    ),

    "lonely": (
        [
            "Loneliness can make the world feel much smaller than it really is.",
            "Being lonely right now does not mean you will always be lonely.",
            "You do not need a huge friend group to feel connected."
        ],
        [
            "Message one person you already feel reasonably comfortable with.",
            "Join a shared-interest activity where conversation happens naturally.",
            "Aim for one genuine interaction instead of trying to become popular.",
            "Spend time in places where you naturally encounter people.",
            "Tell a trusted adult if loneliness is becoming overwhelming."
        ]
    ),

    "overthinking": (
        [
            "Overthinking can feel like problem-solving even when the same thought is simply going in circles.",
            "A possibility is not the same thing as a fact.",
            "You do not have to solve your entire future tonight."
        ],
        [
            "Write the thought down and label it fact, possibility or fear.",
            "Ask what action is actually available right now.",
            "If there is no action available, give yourself permission to pause it.",
            "Avoid repeatedly checking messages or social media for reassurance.",
            "Do something concrete and ordinary to bring attention back to the present."
        ]
    ),

    "future": (
        [
            "Not knowing your entire future is normal, especially while you are still in school.",
            "You do not need a perfect five-year plan to make a good next decision.",
            "Your future is built through many smaller decisions."
        ],
        [
            "Separate decisions that need to happen now from decisions that can wait.",
            "Research real courses and career paths rather than trying to predict everything.",
            "Keep more than one option open while you gather information.",
            "Talk to a teacher, counsellor or trusted adult with relevant experience.",
            "Choose the next useful step rather than demanding certainty."
        ]
    ),

    "career": (
        [
            "Career confusion does not mean you are behind.",
            "You do not need one magical answer immediately.",
            "It is okay for your plan to change as you learn more."
        ],
        [
            "List subjects and activities you genuinely enjoy.",
            "Research the actual courses and requirements for possible paths.",
            "Write down several realistic options instead of forcing one answer.",
            "Talk to people who actually study or work in those fields.",
            "Give yourself a deadline for researching, then choose a next step."
        ]
    ),

    "college": (
        [
            "College applications can make the future feel extremely immediate.",
            "You do not have to solve your whole adult life through one application."
        ],
        [
            "Make a checklist of documents and deadlines.",
            "Handle one application step at a time.",
            "Keep backup options so one result does not feel like the only possible future.",
            "Avoid comparing your entire future with someone else's college announcement.",
            "Ask a trusted adult or counsellor about confusing requirements."
        ]
    ),

    "sleep": (
        [
            "Poor sleep can make every other stressor feel louder.",
            "Rest is not laziness; it is part of being able to function.",
            "You do not have to solve tomorrow's problems from bed."
        ],
        [
            "Keep your wake-up time reasonably consistent.",
            "Reduce stimulating screens before bed.",
            "If you cannot sleep for a long time, switch to something quiet instead of repeatedly checking the clock.",
            "Avoid turning bedtime into another study session.",
            "If sleep problems keep affecting daily life, tell a trusted adult or healthcare professional."
        ]
    ),

    "anxiety": (
        [
            "Anxiety can make a possibility feel like a certainty.",
            "You do not have to obey every alarming thought.",
            "When your body feels stressed, slowing things down can help you think more clearly."
        ],
        [
            "Take a few slow breaths and relax your shoulders.",
            "Write down what you can control and what you cannot.",
            "Reduce immediate stimulation if possible.",
            "Do one ordinary grounding activity such as washing your face, walking or organising your desk.",
            "If anxiety repeatedly interferes with school, sleep or daily life, talk to a trusted adult or professional."
        ]
    ),

    "low_mood": (
        [
            "You do not have to pretend to be cheerful when you are having a low day.",
            "A difficult mood does not automatically predict your future.",
            "You are allowed to need support."
        ],
        [
            "Keep one or two basic routines going.",
            "Talk to someone you trust instead of isolating yourself.",
            "Change your environment for a little while.",
            "Do one small manageable activity rather than demanding a productive day.",
            "If low mood persists or seriously affects daily life, tell a trusted adult or professional."
        ]
    ),

    "family": (
        [
            "Family stress is exhausting because home is also where you are supposed to recover.",
            "You do not have to solve every family problem yourself.",
            "Your wellbeing matters alongside your responsibilities."
        ],
        [
            "Choose a calm time for difficult conversations.",
            "Focus on one specific problem.",
            "Write your thoughts down before talking if that helps.",
            "Ask a trusted adult outside the conflict for perspective.",
            "Take some space after a heated argument before trying to solve it."
        ]
    ),

    "perfectionism": (
        [
            "Perfectionism can make good work feel like failure.",
            "You do not need to make everything flawless to make it valuable.",
            "A finished good piece of work is more useful than a perfect piece that never gets finished."
        ],
        [
            "Define 'good enough' before starting.",
            "Set a time limit for polishing.",
            "Ask whether another hour will actually change the result.",
            "Allow ordinary mistakes while learning.",
            "Finish when the important requirements are met."
        ]
    ),

    "health": (
        [
            "Health worries deserve attention without immediately assuming the worst.",
            "Stress can affect how you feel physically, but persistent symptoms should not simply be dismissed as stress."
        ],
        [
            "Tell a trusted adult if a health concern is worrying you.",
            "Keep a simple note of symptoms and when they occur.",
            "Avoid diagnosing yourself from random internet posts.",
            "Prioritise reasonable sleep, food, hydration and rest.",
            "For severe, persistent or disruptive symptoms, seek appropriate medical care."
        ]
    ),

    "generic": (
        [
            "That sounds like a lot to carry. You do not have to solve everything at once.",
            "A stressful moment can make everything feel urgent, even when only a few things need attention now.",
            "You can take your feelings seriously without treating every stressed thought as a fact."
        ],
        [
            "Write the exact problem in one sentence.",
            "Separate what you can control from what you cannot.",
            "Choose one small action you can complete today.",
            "Take a short break if you are overwhelmed.",
            "Talk to a trusted adult, teacher, counsellor or friend if the problem is becoming too much to carry alone."
        ]
    )
}


KEYWORDS = {
    "exams": ["exam", "exams", "test", "tests", "board", "boards", "preboard", "paper"],
    "marks": ["mark", "marks", "score", "result", "results", "percentage", "grade"],
    "study": ["study", "studying", "syllabus", "revision", "revise", "chapter", "homework"],
    "deadlines": ["deadline", "deadlines", "due", "backlog", "pending", "assignment"],
    "procrastination": ["procrastinat", "delaying", "putting off", "keep delaying", "can't start", "cannot start"],
    "focus": ["focus", "concentrate", "concentration", "distracted", "can't focus", "cannot focus"],
    "memory": ["forget", "forgot", "remember", "memory", "can't remember", "cannot remember"],
    "parents": ["parent", "parents", "mom", "mum", "dad", "mother", "father"],
    "friends": ["friend", "friends", "bestie", "classmate"],
    "bullying": ["bully", "bullied", "bullying", "harass", "harassment"],
    "lonely": ["lonely", "alone", "isolated", "no friends"],
    "overthinking": ["overthink", "overthinking", "can't stop thinking", "cannot stop thinking"],
    "career": ["career", "career choice", "career path"],
    "college": ["college", "university", "admission", "admissions"],
    "future": ["future", "what will happen", "what am i going to do"],
    "sleep": ["sleep", "insomnia", "can't sleep", "cannot sleep", "sleeping late"],
    "anxiety": ["anxiety", "anxious", "panic", "panicking", "nervous", "overwhelmed"],
    "low_mood": ["sad", "sadness", "low mood", "feeling low", "down", "unhappy"],
    "health": ["health", "sick", "ill", "illness", "doctor", "symptom"],
    "family": ["family", "home problem", "problem at home"],
    "perfectionism": ["perfect", "perfectionist", "perfectionism", "never good enough"]
}


def detect_category(message):
    text = message.lower()
    scores = {}

    for category, words in KEYWORDS.items():
        for word in words:
            if word in text:
                scores[category] = scores.get(category, 0) + len(word.split()) + 1

    # Specific situations get priority.
    if any(x in text for x in ["suicide", "kill myself", "want to die", "end my life"]):
        return "safety"

    if "friend" in text or "friends" in text:
        if any(x in text for x in ["mean", "rude", "ignore", "ignoring", "fight", "hurt", "toxic"]):
            return "friends"

    if "parent" in text or "parents" in text or "mom" in text or "dad" in text:
        return "parents"

    if "break" in text and "up" in text:
        return "breakup"

    if "social media" in text or "instagram" in text or "reels" in text:
        return "social"

    return max(scores, key=scores.get) if scores else "generic"


def safety_check(message):
    text = message.lower()

    phrases = [
        "kill myself",
        "want to die",
        "end my life",
        "suicide",
        "self harm",
        "self-harm",
        "hurt myself"
    ]

    return any(p in text for p in phrases)


def safety_response():
    responses = (
        "I'm really sorry you're dealing with something this painful. "
        "Please tell a trusted adult right now, such as a parent, guardian, "
        "teacher, school counsellor or another adult you trust. "
        "Try to stay with someone rather than being alone. "
        "If you are in immediate danger, contact your local emergency service "
        "or go to the nearest emergency department."
    ),

    (
        "I'm really glad you said something instead of keeping this completely "
        "to yourself. Please reach out to a trusted adult right now and tell "
        "them honestly that you're not feeling safe. A parent, guardian, "
        "teacher, counsellor or another trusted adult can stay with you and "
        "help you get through this. If there is immediate danger, contact "
        "your local emergency service or go to the nearest emergency department."
    ),

    (
        "You don't have to handle a moment like this on your own. "
        "Please find a trusted adult and stay with them while you talk about "
        "what you're experiencing. You can start with something as simple as, "
        "\"I'm having a really difficult time and I need you to stay with me.\" "
        "If you're in immediate danger, contact your local emergency service "
        "or go to the nearest emergency department."
    ),

    (
        "What you're going through deserves real support, not something you "
        "have to carry silently. Please tell a parent, guardian, teacher, "
        "school counsellor or another adult you trust and stay around people "
        "who can support you. If you feel that you may be in immediate danger, "
        "contact your local emergency service or go to the nearest emergency "
        "department."
    ),

    (
        "Please pause everything else for a moment and reach out to someone "
        "you trust. You don't need to explain everything perfectly; you can "
        "simply tell them that you're struggling and need them to stay with "
        "you. If you're in immediate danger, contact your local emergency "
        "service or go to the nearest emergency department."
    ),

    (
        "I'm sorry that things feel this overwhelming right now. "
        "Please don't face this alone. Go to a trusted adult, tell them what "
        "you're experiencing, and stay with them. This could be a parent, "
        "guardian, teacher, school counsellor or another adult you trust. "
        "If there is immediate danger, contact your local emergency service "
        "or go to the nearest emergency department."
    ),

    (
        "You deserve support from a real person who can be with you right now. "
        "Please tell a trusted adult how serious things feel and ask them to "
        "stay with you. If talking feels difficult, you can simply show them "
        "this message. If you're in immediate danger, contact your local "
        "emergency service or go to the nearest emergency department."
    ),

    (
        "I'm here to listen, but this is something you shouldn't have to manage "
        "through a chatbot alone. Please reach out to a trusted adult right "
        "now and stay with someone you trust. If you're in immediate danger, "
        "contact your local emergency service or go to the nearest emergency "
        "department."
    )


def basic_reply(message):
    text = message.lower().strip()

    if text in {"hi", "hello", "hey", "hii", "hiii","hie"}:
        return (
            "Hey! I'm Sahaara. Tell me what's stressing you out, "
            "and I'll help you break it down."
        )

    if text in {"bye", "goodbye", "good bye", "see you", "see ya", "cya","byeee","byee"}:
        return (
            "Bye! Take care, and remember to take things one step at a time. "
            "I'm here whenever you need to talk."
        )

    if text in {"thanks", "thank you", "thank u", "thx", "thankyouuuuu", "thankyouuu","thankyouu","thankyou"}:
        return (
            "You're welcome. Take it one step at a time. "
            "You do not have to solve everything at once."
        )

    if text in {"okay", "got it", "i understand", "yessss", "yes fine", "fine"}:
        return (
            "Anything else that bothers you?"
            "Remember you got this!"
        )

    return None


def generate_response(message, stress_level=None, previous_category=None):

    # Safety comes first.
    if safety_check(message):
        return safety_response()

    basic = basic_reply(message)

    if basic:
        return basic

    category = detect_category(message)

    # If the user gives a short follow-up such as
    # "yeah", "exactly", "that's the problem", keep the previous topic.
    if previous_category and category == "generic":
        category = previous_category

    calm, solutions = DATA.get(category, DATA["generic"])

    opening = random.choice(calm)

    # Mention the ML result only when it is useful.
    stress_note = ""

    if stress_level == "High":
        stress_note = (
            " Your stress assessment is currently on the higher side, "
            "so let's focus on reducing the immediate load rather than "
            "trying to fix everything at once."
        )

    elif stress_level == "Moderate":
        stress_note = (
            " Your assessment suggests some elevated stress, so it may "
            "help to deal with the biggest pressure point first."
        )

    # Pick varied suggestions.
    chosen = random.sample(
        solutions,
        min(3, len(solutions))
    )

    endings = [
        "For now, just take the next manageable step.",
        "You do not need to fix everything today.",
        "Let's make the problem smaller before trying to solve all of it.",
        "One difficult moment does not define your whole situation.",
        "Take it one step at a time and let the rest wait for a moment."
    ]

    answer = opening + stress_note

    answer += "\n\nHere are a few things you could try:"

    for i, item in enumerate(chosen, 1):
        answer += f"\n{i}. {item}"

    answer += "\n\n" + random.choice(endings)

    return answer




Writing chatbot.py


In [ ]:
%%writefile app.py

import streamlit as st
import streamlit.components.v1 as components
import pickle
import pandas as pd
import random
import time

# =========================================================
# PAGE SETUP
# =========================================================

st.set_page_config(
    page_title="Pause | Student Wellbeing",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# =========================================================
# LOAD MODELS
# =========================================================

with open("anomaly_model.pkl", "rb") as f:
    anomaly_model = pickle.load(f)

with open("stress_model.pkl", "rb") as f:
    stress_model = pickle.load(f)

from chatbot import generate_response, detect_category, safety_check


# =========================================================
# SESSION STATE
# =========================================================

defaults = {
    "stress_level": None,
    "stress_index": 0,
    "anomaly_result": None,

    "chat_history": [],
    "previous_category": None,

    "selected_theme": "Navy & Mauve",
    "appearance": "🌙 Dark",

    "memory_cards": [],
    "memory_flipped": [],
    "memory_matched": set(),
    "memory_attempts": 0,
    "memory_message": "",
    "memory_started": False,

    "focus_running": False,
    "focus_start_time": None,
    "focus_score": 0,
    "focus_round": 0,
    "focus_position": random.randint(0, 5),
    "focus_finished": False,
}

for key, value in defaults.items():
    if key not in st.session_state:
        st.session_state[key] = value


# =========================================================
# THEMES
# =========================================================

THEMES = {
    "Navy & Mauve": {
        "dark": "linear-gradient(135deg, #171A2B 0%, #25243A 48%, #352B3C 100%)",
        "light": "linear-gradient(135deg, #F7F0EF 0%, #F2E6E7 48%, #E8DDE4 100%)",
        "accent": "#BD8E89",
        "accent_light": "#D9AAA3",
    },

    "Forest": {
        "dark": "linear-gradient(135deg, #16231E 0%, #20372E 50%, #2D4339 100%)",
        "light": "linear-gradient(135deg, #F0F5F0 0%, #E4EEE7 50%, #DCE8DF 100%)",
        "accent": "#7F9D89",
        "accent_light": "#A9C2AF",
    },

    "Lavender": {
        "dark": "linear-gradient(135deg, #211D30 0%, #302942 50%, #3B3150 100%)",
        "light": "linear-gradient(135deg, #F7F2FA 0%, #EEE6F4 50%, #E7DDF0 100%)",
        "accent": "#9B83B5",
        "accent_light": "#BBA8D0",
    },

    "Rose": {
        "dark": "linear-gradient(135deg, #281C22 0%, #3B252F 50%, #492D38 100%)",
        "light": "linear-gradient(135deg, #FBF1F3 0%, #F4E2E7 50%, #ECD8DE 100%)",
        "accent": "#B77C8D",
        "accent_light": "#D49AAA",
    },
}

theme = THEMES[st.session_state.selected_theme]


# =========================================================
# APPEARANCE
# =========================================================

if st.session_state.appearance == "☀️ Light":
    page_bg = theme["light"]
    text_color = "#241E1D"
    card_bg = "rgba(255,255,255,0.72)"
else:
    page_bg = theme["dark"]
    text_color = "#F7F1EF"
    card_bg = "rgba(255,255,255,0.07)"


# =========================================================
# GLOBAL CSS
# =========================================================

st.markdown(
    f"""
<style>

[data-testid="stAppViewContainer"] {{
    background: {page_bg};
    color: {text_color};
}}

[data-testid="stHeader"] {{
    background: transparent;
}}

[data-testid="stSidebar"] {{
    display: none;
}}

.block-container {{
    max-width: 1250px;
    padding-top: 2rem;
    padding-bottom: 4rem;
}}

h1, h2, h3, h4, h5, h6 {{
    color: {text_color} !important;
}}

p, label {{
    color: {text_color};
}}

.hero-box {{
    background: {card_bg};
    border: 1px solid rgba(255,255,255,0.10);
    border-radius: 28px;
    padding: 42px;
    margin: 10px 0 28px 0;
}}

.small-label {{
    color: {theme["accent_light"]};
    font-size: 0.8rem;
    font-weight: 700;
    letter-spacing: 1.5px;
}}

.soft-box {{
    background: {card_bg};
    border: 1px solid rgba(255,255,255,0.10);
    border-radius: 22px;
    padding: 24px;
    margin: 12px 0;
}}

div.stButton > button {{
    border-radius: 14px;
    min-height: 42px;
    font-weight: 600;
}}


/* =========================================================
   SAHAARA
   ========================================================= */

[data-testid="stChatMessage"] {{
    color: #FFFFFF !important;
}}

[data-testid="stChatMessageContent"] {{
    color: #FFFFFF !important;
}}

[data-testid="stChatMessageContent"] p {{
    color: #FFFFFF !important;
}}

[data-testid="stChatMessageContent"] strong {{
    color: #FFFFFF !important;
}}

[data-testid="stChatMessageContent"] li {{
    color: #FFFFFF !important;
}}

[data-testid="stChatMessageContent"] span {{
    color: #FFFFFF !important;
}}

[class*="st-key-sahaara_form"] input {{
    color: #FFFFFF !important;
    -webkit-text-fill-color: #FFFFFF !important;
    caret-color: #FFFFFF !important;
}}

[class*="st-key-sahaara_form"] input::placeholder {{
    color: #FFFFFF !important;
    -webkit-text-fill-color: #FFFFFF !important;
    opacity: 0.65 !important;
}}

</style>
""",
    unsafe_allow_html=True
)


# =========================================================
# SAHAARA FOLLOW-UPS
# =========================================================

FOLLOW_UPS = {

    "exams":
        "What feels heavier right now: the amount you have to study, or the fear of how you'll perform?",

    "marks":
        "Is it the marks themselves bothering you, or what you feel they say about you?",

    "study":
        "Which subject or topic is taking up most of your mental space right now?",

    "deadlines":
        "What's the one deadline that feels most urgent at the moment?",

    "procrastination":
        "When you put the work off, is it usually because you're tired, overwhelmed, or don't know where to start?",

    "focus":
        "What usually breaks your concentration first: your phone, thoughts, tiredness, or something else?",

    "memory":
        "Are you struggling more with remembering things, or with understanding them in the first place?",

    "parents":
        "Do you feel like you can actually explain what you're going through to them?",

    "friends":
        "Do you want help with what happened, or mostly want somewhere to talk about it?",

    "bullying":
        "Is this something that happens repeatedly, or was there one particular incident?",

    "lonely":
        "Do you feel lonely even when you're around people, or are you actually spending a lot of time alone?",

    "overthinking":
        "Is there one particular thought that keeps coming back?",

    "future":
        "Is the uncertainty itself bothering you, or are you worried about choosing the wrong path?",

    "career":
        "Are you unsure about what you want, or do you already have an idea but feel pressured about it?",

    "college":
        "What's stressing you most about college right now?",

    "sleep":
        "Has your sleep been affected more by your schedule or by your thoughts at night?",

    "anxiety":
        "What tends to make that feeling stronger for you?",

    "low_mood":
        "Has this been more of a today problem, or has it been sitting with you for a while?",

    "family":
        "Do you want to talk about what happened, or figure out what you want to do next?",

    "perfectionism":
        "What feels like it would happen if you didn't do something perfectly?",
}


# =========================================================
# SAHAARA CHAT
# =========================================================

with st.popover("💬 Sahaara"):

    st.markdown("### 💬 Sahaara")

    st.caption(
        "A quiet space to talk things through."
    )

    if not st.session_state.chat_history:

        st.info(
            "You can talk about school, stress, friendships, "
            "future plans, or anything that's weighing on you."
        )

    for role, message in st.session_state.chat_history:

        with st.chat_message(role):
            st.write(message)

    with st.form(
        "sahaara_form",
        clear_on_submit=True
    ):

        user_message = st.text_input(
            "Message Sahaara",
            placeholder="What's on your mind?"
        )

        send = st.form_submit_button("Send")

        if send and user_message.strip():

            message = user_message.strip()

            st.session_state.chat_history.append(
                ("user", message)
            )

            if safety_check(message):

                response = (
                    "I'm really glad you said something. "
                    "Please tell a trusted adult who can stay with you "
                    "and help you get support right now."
                )

                category = "generic"

            else:

                category = detect_category(message)

                response = generate_response(
                    message,
                    st.session_state.stress_level,
                    st.session_state.previous_category
                )

                if category in FOLLOW_UPS:

                    response += (
                        "\n\n" +
                        FOLLOW_UPS[category]
                    )

            st.session_state.previous_category = category

            st.session_state.chat_history.append(
                ("assistant", response)
            )

            st.rerun()


# =========================================================
# MAIN TABS
# =========================================================

tab_home, tab_ground, tab_daily, tab_comfort = st.tabs(
    [
        "🏠 Home",
        "🌿 Ground Reset",
        "📊 Daily Check-In",
        "🎮 Comfort Corner",
    ]
)


# =========================================================
# HOME
# =========================================================

with tab_home:

    st.markdown(
        '<div class="hero-box">',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="small-label">PAUSE</div>',
        unsafe_allow_html=True
    )

    st.title(
        "You don't have to have everything figured out."
    )

    st.write(
        "A student wellbeing space designed to help you "
        "pause, understand what you're feeling, and take "
        "one manageable step at a time."
    )

    st.markdown(
        "</div>",
        unsafe_allow_html=True
    )

    col1, col2 = st.columns(2)

    with col1:

        st.markdown(
            '<div class="soft-box">',
            unsafe_allow_html=True
        )

        st.subheader("🌿 Ground Reset")

        st.write(
            "Use breathing, grounding, or a thought dump "
            "when your mind feels overloaded."
        )

        st.markdown(
            "</div>",
            unsafe_allow_html=True
        )

    with col2:

        st.markdown(
            '<div class="soft-box">',
            unsafe_allow_html=True
        )

        st.subheader("📊 Daily Check-In")

        st.write(
            "Reflect on sleep, workload, mood, social time, "
            "screen time and other everyday factors."
        )

        st.markdown(
            "</div>",
            unsafe_allow_html=True
        )

    st.markdown("---")

    st.subheader("🎨 Personalise your space")

    col1, col2 = st.columns(2)

    with col1:

        selected_theme = st.selectbox(
            "Theme",
            list(THEMES.keys()),
            index=list(THEMES.keys()).index(
                st.session_state.selected_theme
            )
        )

        if selected_theme != st.session_state.selected_theme:

            st.session_state.selected_theme = selected_theme
            st.rerun()

    with col2:

        appearance = st.selectbox(
            "Appearance",
            ["🌙 Dark", "☀️ Light"],
            index=[
                "🌙 Dark",
                "☀️ Light"
            ].index(st.session_state.appearance)
        )

        if appearance != st.session_state.appearance:

            st.session_state.appearance = appearance
            st.rerun()

    st.markdown("---")

    st.info(
        "💬 Sahaara is available from the button above "
        "whenever you want to talk something through."
    )


# =========================================================
# GROUND RESET
# =========================================================

with tab_ground:

    st.header("🌿 Ground Reset")

    st.write(
        "A small pause for moments when your mind feels too full."
    )

    ground_reset_html = """
<!DOCTYPE html>

<html>

<head>

<style>

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    padding: 2px;
    font-family: Arial, sans-serif;
    background: transparent;
    color: #2B2524;
}

.container {
    width: 100%;
}

.card {
    background: rgba(255,255,255,0.80);
    border: 1px solid rgba(100,80,80,0.15);
    border-radius: 18px;
    padding: 16px;
    margin-bottom: 12px;
    box-shadow: 0 4px 14px rgba(50,30,30,0.05);
}

.card h3 {
    margin: 0 0 7px 0;
    color: #3B3030;
    font-size: 18px;
}

.card p {
    color: #5A4D4D;
    line-height: 1.35;
    margin: 6px 0;
    font-size: 14px;
}


/* BREATHING */

.breath-area {
    text-align: center;
    padding: 2px 0;
}

.breath-circle {
    width: 90px;
    height: 90px;
    border-radius: 50%;
    background: linear-gradient(
        135deg,
        #BD8E89,
        #9E7370
    );
    margin: 12px auto;
    display: flex;
    align-items: center;
    justify-content: center;
    color: #FFFFFF;
    font-size: 14px;
    font-weight: 700;
    box-shadow:
        0 0 0 0 rgba(189,142,137,0.35),
        0 6px 18px rgba(100,60,60,0.18);
    transform: scale(1);
}

.breath-circle.breathe-in {
    animation: breatheIn 4s ease-in-out forwards;
}

.breath-circle.breathe-out {
    animation: breatheOut 4s ease-in-out forwards;
}

@keyframes breatheIn {
    0% {
        transform: scale(0.82);
    }

    100% {
        transform: scale(1.25);
    }
}

@keyframes breatheOut {
    0% {
        transform: scale(1.25);
    }

    100% {
        transform: scale(0.82);
    }
}


/* BUTTONS */

button {
    border: 1px solid #BD8E89;
    border-radius: 10px;
    padding: 9px 16px;
    cursor: pointer;
    font-weight: 700;
    font-size: 13px;
    background: #BD8E89;
    color: #FFFFFF;
    box-shadow: 0 3px 9px rgba(0,0,0,0.10);
    transition:
        transform 0.15s ease,
        background 0.15s ease;
}

button:hover {
    background: #7F6269;
    color: #FFFFFF;
    transform: translateY(-1px);
}

button:active {
    transform: translateY(0);
}


/* GROUNDING */

.ground-grid {
    display: grid;
    grid-template-columns: repeat(5, 1fr);
    gap: 7px;
    margin-top: 10px;
}

.sense {
    background: rgba(189,142,137,0.13);
    border-radius: 11px;
    padding: 9px 5px;
    text-align: center;
    color: #5A4D4D;
    font-size: 12px;
    line-height: 1.25;
}

.sense strong {
    display: block;
    color: #3B3030;
    font-size: 19px;
    margin-bottom: 2px;
}


/* THOUGHT DUMP */

textarea {
    width: 100%;
    min-height: 100px;
    border-radius: 11px;
    border: 1px solid rgba(100,80,80,0.25);
    padding: 11px;
    font-family: Arial, sans-serif;
    font-size: 13px;
    resize: none;
    color: #2B2524;
    background: rgba(255,255,255,0.92);
}

textarea::placeholder {
    color: #7B6A68;
}

.thought-actions {
    display: flex;
    gap: 9px;
    margin-top: 9px;
    flex-wrap: wrap;
}


/* ANIMATION */

.animation-box {
    display: none;
    margin-top: 10px;
    padding: 10px;
    border-radius: 13px;
    text-align: center;
    background: rgba(189,142,137,0.08);
    overflow: visible;
    min-height: 105px;
}

.animation-box.active {
    display: block;
}

.paper {
    width: 150px;
    min-height: 55px;
    margin: 0 auto 5px;
    padding: 10px;
    border-radius: 6px;
    background: #FFFDF8;
    color: #4D403E;
    font-size: 12px;
    box-shadow: 0 4px 10px rgba(0,0,0,0.12);
    position: relative;
}

.paper.burning {
    animation: burnPaper 2.2s ease-in forwards;
}

@keyframes burnPaper {

    0% {
        transform: translateY(0) rotate(0deg) scale(1);
        opacity: 1;
        filter: brightness(1);
    }

    30% {
        transform: translateY(-4px) rotate(-2deg) scale(1);
        opacity: 1;
        filter: brightness(1.1) sepia(0.2);
    }

    60% {
        transform: translateY(-12px) rotate(3deg) scale(0.85);
        opacity: 0.7;
        filter: brightness(1.3) sepia(0.55);
    }

    100% {
        transform: translateY(-30px) rotate(8deg) scale(0.35);
        opacity: 0;
        filter: brightness(1.7) sepia(1);
    }
}

.flames {
    height: 24px;
    margin-top: -4px;
    font-size: 20px;
    opacity: 0;
}

.flames.show {
    animation: flameShow 2.1s ease-in-out forwards;
}

@keyframes flameShow {

    0% {
        opacity: 0;
        transform: scale(0.7);
    }

    20% {
        opacity: 1;
        transform: scale(1);
    }

    65% {
        opacity: 1;
        transform: scale(1.08);
    }

    100% {
        opacity: 0;
        transform: scale(0.7);
    }
}


/* SHRED */

.shred-paper {
    width: 150px;
    min-height: 55px;
    margin: 0 auto 5px;
    padding: 10px;
    border-radius: 6px;
    background: #FFFDF8;
    color: #4D403E;
    font-size: 12px;
    box-shadow: 0 4px 10px rgba(0,0,0,0.12);
}

.shred-paper.shredding {
    animation: shredPaper 1.9s ease-in forwards;
}

@keyframes shredPaper {

    0% {
        transform: translateX(0) rotate(0deg) scaleY(1);
        opacity: 1;
        clip-path: inset(0 0 0 0);
    }

    30% {
        transform: translateX(-5px) rotate(-2deg);
        opacity: 0.9;
    }

    60% {
        transform: translateX(7px) rotate(3deg) scaleY(0.65);
        opacity: 0.55;
        clip-path: inset(0 20% 0 20%);
    }

    100% {
        transform: translateX(10px) rotate(6deg) scaleY(0.2);
        opacity: 0;
        clip-path: inset(0 50% 0 50%);
    }
}

.status {
    margin-top: 6px;
    font-weight: 600;
    font-size: 12px;
    color: #7F6269;
    min-height: 16px;
}

@media (max-width: 800px) {

    .ground-grid {
        grid-template-columns: repeat(2, 1fr);
    }

    .thought-actions {
        flex-direction: row;
    }

    .thought-actions button {
        flex: 1;
    }

}

</style>

</head>

<body>

<div class="container">


<div class="card">

    <h3>🌬️ One-Minute Breathing Reset</h3>

    <p>
        Follow the circle. Breathe in as it expands
        and breathe out as it contracts.
    </p>

    <div class="breath-area">

        <div
            id="breathCircle"
            class="breath-circle"
        >
            Ready
        </div>

        <button onclick="startBreathing()">
            Start Breathing
        </button>

        <div
            id="breathStatus"
            class="status"
        ></div>

    </div>

</div>


<div class="card">

    <h3>🌱 5-4-3-2-1 Grounding</h3>

    <p>
        Notice the world around you without judging
        what you're feeling.
    </p>

    <div class="ground-grid">

        <div class="sense">
            <strong>5</strong>
            See
        </div>

        <div class="sense">
            <strong>4</strong>
            Touch
        </div>

        <div class="sense">
            <strong>3</strong>
            Hear
        </div>

        <div class="sense">
            <strong>2</strong>
            Smell
        </div>

        <div class="sense">
            <strong>1</strong>
            Taste
        </div>

    </div>

</div>


<div class="card">

    <h3>📝 Thought Dump</h3>

    <p>
        Write whatever is taking up space in your head.
    </p>

    <textarea
        id="thoughtBox"
        placeholder="Write it all out here..."
    ></textarea>

    <div class="thought-actions">

        <button onclick="burnThoughts()">
            🔥 Burn
        </button>

        <button onclick="shredThoughts()">
            ✂️ Shred
        </button>

    </div>

    <div
        id="animationBox"
        class="animation-box"
    >

        <div
            id="paperAnimation"
            class="paper"
        >
            Your thought
        </div>

        <div
            id="flames"
            class="flames"
        >
            🔥 🔥 🔥
        </div>

        <div
            id="animationStatus"
            class="status"
        ></div>

    </div>

    <div
        id="thoughtStatus"
        class="status"
    ></div>

</div>


</div>


<script>

function startBreathing() {

    const circle =
        document.getElementById("breathCircle");

    const status =
        document.getElementById("breathStatus");

    let round = 0;

    const totalRounds = 7;

    function breathe() {

        if (round >= totalRounds) {

            circle.className =
                "breath-circle";

            circle.innerText =
                "Done";

            status.innerText =
                "🌿 You made space for one quiet minute.";

            return;
        }

        round++;

        circle.className =
            "breath-circle breathe-in";

        circle.innerText =
            "Breathe in";

        status.innerText =
            "Slowly breathe in...";

        setTimeout(function() {

            circle.className =
                "breath-circle breathe-out";

            circle.innerText =
                "Breathe out";

            status.innerText =
                "Slowly breathe out...";

        }, 4000);

        setTimeout(function() {

            breathe();

        }, 8000);
    }

    status.innerText =
        "Take it slowly. There is nowhere else to be.";

    breathe();
}


function burnThoughts() {

    const box =
        document.getElementById("thoughtBox");

    const animationBox =
        document.getElementById("animationBox");

    const paper =
        document.getElementById("paperAnimation");

    const flames =
        document.getElementById("flames");

    const animationStatus =
        document.getElementById("animationStatus");

    const thoughtStatus =
        document.getElementById("thoughtStatus");

    const thought =
        box.value.trim();

    if (!thought) {

        thoughtStatus.innerText =
            "Write something first.";

        return;
    }

    animationBox.classList.add("active");

    paper.className =
        "paper";

    flames.className =
        "flames";

    paper.innerText =
        thought.length > 70
        ? thought.substring(0, 70) + "..."
        : thought;

    animationStatus.innerText =
        "🔥 Letting this thought go...";

    void paper.offsetWidth;
    void flames.offsetWidth;

    paper.classList.add("burning");
    flames.classList.add("show");

    setTimeout(function() {

        box.value = "";

        animationStatus.innerText =
            "✨ It's okay to let some thoughts leave.";

    }, 2200);
}


function shredThoughts() {

    const box =
        document.getElementById("thoughtBox");

    const animationBox =
        document.getElementById("animationBox");

    const paper =
        document.getElementById("paperAnimation");

    const flames =
        document.getElementById("flames");

    const animationStatus =
        document.getElementById("animationStatus");

    const thoughtStatus =
        document.getElementById("thoughtStatus");

    const thought =
        box.value.trim();

    if (!thought) {

        thoughtStatus.innerText =
            "Write something first.";

        return;
    }

    animationBox.classList.add("active");

    flames.className =
        "flames";

    paper.className =
        "shred-paper";

    paper.innerText =
        thought.length > 70
        ? thought.substring(0, 70) + "..."
        : thought;

    animationStatus.innerText =
        "✂️ Shredding that thought...";

    void paper.offsetWidth;

    paper.classList.add("shredding");

    setTimeout(function() {

        box.value = "";

        animationStatus.innerText =
            "✨ That thought doesn't have to stay with you.";

    }, 1900);
}

</script>

</body>

</html>
"""

    components.html(
        ground_reset_html,
        height=760,
        scrolling=False
    )


# =========================================================
# DAILY CHECK-IN
# =========================================================

with tab_daily:

    st.header("📊 Daily Check-In")

    st.write(
        "Answer based on how today has actually been."
    )

    col1, col2 = st.columns(2)

    with col1:

        sleep_hrs = st.slider(
            "Sleep hours",
            2.0,
            12.0,
            7.0,
            0.5
        )

        sleep_quality = st.slider(
            "Sleep quality",
            1,
            5,
            3
        )

        exercise_mins = st.slider(
            "Exercise / movement minutes",
            0,
            180,
            30,
            10
        )

        academic_hrs = st.slider(
            "Academic hours",
            0.0,
            14.0,
            6.0,
            0.5
        )

        workload_pressure = st.slider(
            "Workload pressure",
            1,
            5,
            3
        )

    with col2:

        caffeine_mg = st.slider(
            "Caffeine intake (mg)",
            0,
            800,
            150,
            50
        )

        screen_hrs = st.slider(
            "Screen time",
            0.0,
            14.0,
            4.0,
            0.5
        )

        mood_rating = st.slider(
            "Mood today",
            1,
            5,
            3
        )

        social_hrs = st.slider(
            "Social time",
            0.0,
            30.0,
            8.0,
            1.0
        )

        social_quality = st.slider(
            "Quality of social connection",
            1,
            5,
            3
        )

    st.markdown("---")

    if st.button(
        "🔎 Run AI Factor Analysis",
        use_container_width=True
    ):

        feature_names = [
            "sleep_hrs",
            "sleep_quality",
            "academic_hrs",
            "caffeine_mg",
            "screen_hrs",
            "exercise_mins",
            "mood_rating",
            "social_hrs",
            "social_quality",
            "workload_pressure",
        ]

        input_data = pd.DataFrame(
            [[
                sleep_hrs,
                sleep_quality,
                academic_hrs,
                caffeine_mg,
                screen_hrs,
                exercise_mins,
                mood_rating,
                social_hrs,
                social_quality,
                workload_pressure,
            ]],
            columns=feature_names
        )

        prediction = stress_model.predict(
            input_data
        )[0]

        anomaly = anomaly_model.predict(
            input_data
        )[0]

        stress_index = {
            "Low": random.randint(15, 35),
            "Moderate": random.randint(36, 65),
            "High": random.randint(66, 90),
        }.get(prediction, 50)

        st.session_state.stress_level = prediction
        st.session_state.stress_index = stress_index
        st.session_state.anomaly_result = anomaly

        st.success(
            "Analysis complete."
        )

    if st.session_state.stress_level:

        level = st.session_state.stress_level
        index = st.session_state.stress_index
        anomaly = st.session_state.anomaly_result

        st.markdown("---")

        st.markdown(
            "### 📊 Your Current Model Result"
        )

        st.markdown(
            "**EXPERIMENTAL MODEL**"
        )

        st.subheader(
            level.upper()
        )

        st.write(
            f"Current model stress index: **{index}/100**"
        )

        st.progress(
            min(index, 100) / 100
        )

        if anomaly == -1:

            st.warning(
                "Today's combination of inputs looked "
                "somewhat unusual compared with the model's "
                "training patterns."
            )

        else:

            st.info(
                "Today's input pattern did not appear unusual "
                "to the model's anomaly detector."
            )

        st.caption(
            "This is an experimental wellbeing model, "
            "not a medical diagnosis."
        )

        st.markdown(
            "### Factors in today's check-in"
        )

        factor_data = pd.DataFrame(
            {
                "Factor": [
                    "Sleep",
                    "Sleep quality",
                    "Academic load",
                    "Caffeine",
                    "Screen time",
                    "Exercise",
                    "Mood",
                    "Social time",
                    "Social quality",
                    "Workload",
                ],

                "Value": [
                    sleep_hrs,
                    sleep_quality,
                    academic_hrs,
                    caffeine_mg / 100,
                    screen_hrs,
                    exercise_mins / 10,
                    mood_rating,
                    social_hrs / 2,
                    social_quality,
                    workload_pressure,
                ],
            }
        ).set_index("Factor")

        st.bar_chart(
            factor_data
        )


# =========================================================
# COMFORT CORNER
# =========================================================

with tab_comfort:

    st.header("🎮 Comfort Corner")

    st.write(
        "Small things to help you pause, reset, or simply "
        "give your brain a break."
    )


    # =====================================================
    # MEMORY MATCH
    # =====================================================

    st.subheader("🌸 Memory Match")

    if not st.session_state.memory_cards:

        symbols = [
            "🌸",
            "⭐",
            "🌙",
            "🍀"
        ]

        cards = symbols * 2

        random.shuffle(cards)

        st.session_state.memory_cards = cards
        st.session_state.memory_flipped = []
        st.session_state.memory_matched = set()
        st.session_state.memory_attempts = 0
        st.session_state.memory_message = ""
        st.session_state.memory_started = True

    cards = st.session_state.memory_cards

    cols = st.columns(4)

    for i, symbol in enumerate(cards):

        with cols[i % 4]:

            is_flipped = (
                i in st.session_state.memory_flipped
                or i in st.session_state.memory_matched
            )

            if i in st.session_state.memory_matched:

                label = symbol

            elif is_flipped:

                label = symbol

            else:

                label = "?"

            disabled = (
                i in st.session_state.memory_matched
                or i in st.session_state.memory_flipped
                or len(st.session_state.memory_flipped) >= 2
            )

            if st.button(
                label,
                key=f"memory_card_{i}",
                use_container_width=True,
                disabled=disabled
            ):

                st.session_state.memory_flipped.append(i)

                if len(st.session_state.memory_flipped) == 2:

                    first, second = (
                        st.session_state.memory_flipped
                    )

                    st.session_state.memory_attempts += 1

                    if cards[first] == cards[second]:

                        st.session_state.memory_matched.add(
                            first
                        )

                        st.session_state.memory_matched.add(
                            second
                        )

                        st.session_state.memory_message = (
                            "✨ Match!"
                        )

                        st.session_state.memory_flipped = []

                    else:

                        st.session_state.memory_message = (
                            "Not a match yet. Press Continue."
                        )

    if st.session_state.memory_message:

        st.info(
            st.session_state.memory_message
        )

    if len(st.session_state.memory_flipped) == 2:

        if st.button(
            "Continue",
            key="memory_continue"
        ):

            st.session_state.memory_flipped = []

            st.session_state.memory_message = ""

            st.rerun()

    st.write(
        f"Attempts: {st.session_state.memory_attempts}"
    )

    if len(st.session_state.memory_matched) == len(cards):

        st.success(
            "🌸 You found all the pairs!"
        )

        if st.button(
            "Play Again",
            key="memory_play_again"
        ):

            symbols = [
                "🌸",
                "⭐",
                "🌙",
                "🍀"
            ]

            new_cards = symbols * 2

            random.shuffle(new_cards)

            st.session_state.memory_cards = new_cards
            st.session_state.memory_flipped = []
            st.session_state.memory_matched = set()
            st.session_state.memory_attempts = 0
            st.session_state.memory_message = ""

            st.rerun()

    else:

        if st.button(
            "Restart Memory Match",
            key="memory_restart"
        ):

            symbols = [
                "🌸",
                "⭐",
                "🌙",
                "🍀"
            ]

            new_cards = symbols * 2

            random.shuffle(new_cards)

            st.session_state.memory_cards = new_cards
            st.session_state.memory_flipped = []
            st.session_state.memory_matched = set()
            st.session_state.memory_attempts = 0
            st.session_state.memory_message = ""

            st.rerun()


    # =====================================================
    # FOCUS CHALLENGE
    # =====================================================

    st.markdown("---")

    st.subheader("🎯 Focus Challenge")

    st.write(
        "Find the target as quickly as you can."
    )

    if not st.session_state.focus_running:

        if not st.session_state.focus_finished:

            if st.button(
                "Start Focus Challenge",
                key="focus_start"
            ):

                st.session_state.focus_running = True
                st.session_state.focus_start_time = time.time()
                st.session_state.focus_score = 0
                st.session_state.focus_round = 0
                st.session_state.focus_position = random.randint(
                    0,
                    5
                )

                st.rerun()

        else:

            st.success(
                f"Challenge complete! "
                f"Your score: {st.session_state.focus_score}"
            )

            if st.button(
                "Play Again",
                key="focus_again"
            ):

                st.session_state.focus_running = True
                st.session_state.focus_finished = False
                st.session_state.focus_start_time = time.time()
                st.session_state.focus_score = 0
                st.session_state.focus_round = 0
                st.session_state.focus_position = random.randint(
                    0,
                    5
                )

                st.rerun()

    else:

        elapsed = (
            time.time()
            - st.session_state.focus_start_time
        )

        remaining = max(
            0,
            30 - int(elapsed)
        )

        st.write(
            f"⏱️ Time remaining: **{remaining} seconds**"
        )

        cols = st.columns(6)

        for i in range(6):

            with cols[i]:

                if i == st.session_state.focus_position:

                    button_label = "🎯"

                else:

                    button_label = "•"

                if st.button(
                    button_label,
                    key=f"focus_{st.session_state.focus_round}_{i}",
                    use_container_width=True
                ):

                    if i == st.session_state.focus_position:

                        st.session_state.focus_score += 1

                        st.session_state.focus_round += 1

                        st.session_state.focus_position = random.randint(
                            0,
                            5
                        )

                    st.rerun()

        if elapsed >= 30:

            st.session_state.focus_running = False
            st.session_state.focus_finished = True

            st.rerun()

        st.write(
            f"Score: **{st.session_state.focus_score}**"
        )

        if st.button(
            "Restart Challenge",
            key="focus_restart"
        ):

            st.session_state.focus_running = True
            st.session_state.focus_finished = False
            st.session_state.focus_start_time = time.time()
            st.session_state.focus_score = 0
            st.session_state.focus_round = 0
            st.session_state.focus_position = random.randint(
                0,
                5
            )

            st.rerun()


    # =====================================================
    # LITTLE THINGS THAT HELP
    # =====================================================

    st.markdown("---")

    st.subheader("🌷 Little Things That Help")

    st.write(
        "You don't always have to fix the problem. "
        "Sometimes you just need a small pause."
    )

    video1, video2, video3 = st.columns(3)


    # -----------------------------------------------------
    # PUPPY
    # -----------------------------------------------------

    with video1:

        st.markdown("### 🐶 Puppy Therapy")

        st.caption(
            "A little dose of puppy chaos."
        )

        st.video(
            "https://www.youtube.com/watch?v=sYaq3WWyW8w"
        )


    # -----------------------------------------------------
    # BABY
    # -----------------------------------------------------

    with video2:

        st.markdown("### 👶 Baby Giggles")

        st.caption(
            "Wholesome baby moments for a quick reset."
        )

        st.video(
            "https://www.youtube.com/watch?v=8PHbr2m7auo"
        )


    # -----------------------------------------------------
    # ASMR
    # -----------------------------------------------------

    with video3:

        st.markdown("### 🎧 ASMR Reset")

        st.caption(
            "Relaxing sounds when you just need to slow down."
        )

        st.video(
            "https://www.youtube.com/watch?v=TTXcHEMfLb4"
        )

Writing app.py


In [ ]:
!pkill -f streamlit || true

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

import time
time.sleep(5)

print(open("streamlit.log").read())

^C


2026-10-04 06:04:53.107 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.70.85.160:8501




In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb

!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.3) ...
Setting up cloudflared (2026.9.3) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-10-04T06:05:50Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-04T06:05:50Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-04T06:05:54Z INF +--------------------------------------------------------------------------------------------+
2026-10-04T06:05:54Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-04T06:05:54Z INF |  https://aimed-ethernet-virgin-sends.trycloudflare.com